In [ ]:
import os

# Check what folders actually exist
print("Checking your data folder...")
base = '/content/brain_tumor_data'

if os.path.exists(base):
    print(f"✅ Found: {base}")
    print("Contents:")
    for item in os.listdir(base):
        print(f"   📁 {item}")
else:
    print(f"❌ NOT FOUND: {base}")
    print("You need to extract the zip file first.")

In [ ]:
import zipfile
import os

# Path to your zip file in Google Drive
zip_path = '/content/drive/MyDrive/brain_tumor_research/archive.zip'
# Where to extract it
extract_path = '/content/brain_tumor_data'

# Create folder and extract
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Data extracted!")
print("Contents:")
for item in os.listdir(extract_path):
    print(f"   📁 {item}")

In [ ]:
import os

# Search for your data in Google Drive
drive_path = '/content/drive/MyDrive'

# First, check if Drive is mounted
if not os.path.exists(drive_path):
    print("❌ Google Drive NOT mounted!")
    print("Run this first:")
    print("   from google.colab import drive")
    print("   drive.mount('/content/drive')")
else:
    print("✅ Google Drive mounted")
    print("\n🔍 Searching for 'brain_tumor' or 'archive.zip'...")

    found = []
    for root, dirs, files in os.walk(drive_path):
        # Limit depth to avoid scanning forever
        depth = root.replace(drive_path, '').count(os.sep)
        if depth > 5:
            continue

        for f in files:
            if 'brain' in f.lower() or 'tumor' in f.lower() or 'archive' in f.lower():
                if f.endswith('.zip'):
                    found.append(os.path.join(root, f))
                    print(f"   📦 ZIP: {os.path.join(root, f)}")

        for d in dirs:
            if 'brain' in d.lower() or 'tumor' in d.lower():
                full = os.path.join(root, d)
                print(f"   📁 FOLDER: {full}")
                # Show contents
                try:
                    items = os.listdir(full)[:5]
                    for item in items:
                        print(f"      └─ {item}")
                except:
                    pass

    if not found:
        print("\n❌ Nothing found with 'brain' or 'tumor' in the name.")
        print("Tell me: what is your zip file called and where did you put it in Drive?")


In [ ]:
import os

# Search for your data in Google Drive
drive_path = '/content/drive/MyDrive'

# First, check if Drive is mounted
if not os.path.exists(drive_path):
    print("❌ Google Drive NOT mounted!")
    print("Run this first:")
    print("   from google.colab import drive")
    print("   drive.mount('/content/drive')")
else:
    print("✅ Google Drive mounted")
    print("\n🔍 Searching for 'brain_tumor' or 'archive.zip'...")

    found = []
    for root, dirs, files in os.walk(drive_path):
        # Limit depth to avoid scanning forever
        depth = root.replace(drive_path, '').count(os.sep)
        if depth > 5:
            continue

        for f in files:
            if 'brain' in f.lower() or 'tumor' in f.lower() or 'archive' in f.lower():
                if f.endswith('.zip'):
                    found.append(os.path.join(root, f))
                    print(f"   📦 ZIP: {os.path.join(root, f)}")

        for d in dirs:
            if 'brain' in d.lower() or 'tumor' in d.lower():
                full = os.path.join(root, d)
                print(f"   📁 FOLDER: {full}")
                # Show contents
                try:
                    items = os.listdir(full)[:5]
                    for item in items:
                        print(f"      └─ {item}")
                except:
                    pass

    if not found:
        print("\n❌ Nothing found with 'brain' or 'tumor' in the name.")
        print("Tell me: what is your zip file called and where did you put it in Drive?")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/brain_tumor_research/archive.zip'
extract_path = '/content/brain_tumor_data'

if os.path.exists(zip_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Extracted! Contents:")
    for item in os.listdir(extract_path):
        print(f"   {item}")
else:
    print("❌ Zip not found at expected path.")
    print("In Colab, click the 📁 Files icon on the LEFT sidebar.")
    print("Navigate to where your zip is, right-click it → Copy path, and paste that path here.")

✅ Extracted! Contents:
   Training
   Testing


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score
import os

# ========== PATHS ==========
TRAIN_DIR = '/content/brain_tumor_data/Training'
TEST_DIR = '/content/brain_tumor_data/Testing'

# ========== CONFIG ==========
CONFIG = {
    'img_size': (224, 224),
    'batch_size': 32,
    'epochs': 30,
    'lr': 1e-4,
    'seeds': [42, 123, 456],
}

PREPROCESS = {
    'mobilenetv2': mobilenet_preprocess,
    'resnet50': resnet_preprocess,
    'efficientnetb0': efficientnet_preprocess,
}

# ========== MODEL BUILDER ==========
def build_model(name, seed):
    tf.random.set_seed(seed)
    np.random.seed(seed)

    inp = layers.Input(shape=(224, 224, 3))

    if name == 'mobilenetv2':
        base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
    elif name == 'efficientnetb0':
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)

    base.trainable = True
    for layer in base.layers[:-50]:
        layer.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(4, activation='softmax')(x)

    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=CONFIG['lr']),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ========== GENERATORS ==========
def make_gens(name):
    pre = PREPROCESS[name]

    train_aug = ImageDataGenerator(
        preprocessing_function=pre, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15,
        zoom_range=0.15, horizontal_flip=True,
        brightness_range=[0.8, 1.2], validation_split=0.2)

    test_gen = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
        TEST_DIR, target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
        class_mode='categorical', shuffle=False)

    train_gen = train_aug.flow_from_directory(
        TRAIN_DIR, target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
        class_mode='categorical', subset='training', seed=42)

    val_gen = train_aug.flow_from_directory(
        TRAIN_DIR, target_size=CONFIG['img_size'], batch_size=CONFIG['batch_size'],
        class_mode='categorical', subset='validation', shuffle=False, seed=42)

    return train_gen, val_gen, test_gen

# ========== TRAIN & EVALUATE ==========
def run_model(name):
    print(f"\n{'='*60}")
    print(f"MODEL: {name.upper()}")
    print(f"{'='*60}")

    results = []
    for seed in CONFIG['seeds']:
        print(f"\n--- Seed {seed} ---")

        model = build_model(name, seed)
        train_gen, val_gen, test_gen = make_gens(name)

        cb = [
            keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10,
                                          restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=4, verbose=0)
        ]

        model.fit(train_gen, validation_data=val_gen, epochs=CONFIG['epochs'],
                  callbacks=cb, verbose=1)

        # Test eval
        test_gen.reset()
        loss, acc = model.evaluate(test_gen, verbose=0)

        test_gen.reset()
        y_pred = np.argmax(model.predict(test_gen, verbose=0), axis=1)
        y_true = test_gen.classes

        f1 = f1_score(y_true, y_pred, average='weighted')

        results.append({'seed': seed, 'acc': acc*100, 'f1': f1*100})
        print(f"Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}%")

    accs = [r['acc'] for r in results]
    f1s = [r['f1'] for r in results]

    print(f"\n>>> SUMMARY {name.upper()}")
    print(f"Accuracy: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")
    print(f"F1-Score: {np.mean(f1s):.2f}% ± {np.std(f1s):.2f}%")
    print(f"Range: [{np.min(accs):.2f}%, {np.max(accs):.2f}%]")

    return results

# ========== RUN ALL ==========
all_results = {}
for model_name in ['mobilenetv2', 'resnet50', 'efficientnetb0']:
    all_results[model_name] = run_model(model_name)

# ========== FINAL TABLE ==========
print("\n\n" + "="*70)
print("PAPER TABLE — Multi-Seed Results (Mean ± Std)")
print("="*70)
print(f"{'Model':<20} {'Accuracy':<25} {'F1-Score':<25}")
print("-"*70)
for name, res in all_results.items():
    accs = [r['acc'] for r in res]
    f1s = [r['f1'] for r in res]
    print(f"{name:<20} {np.mean(accs):.2f}±{np.std(accs):.2f}%         {np.mean(f1s):.2f}±{np.std(f1s):.2f}%")

print("\n✅ DONE. Copy the table above into your paper.")


MODEL: MOBILENETV2

--- Seed 42 ---


/tmp/ipykernel_1189/3062898614.py:40: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)


Found 1600 images belonging to 4 classes.
Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Epoch 1/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 504s 3s/step - accuracy: 0.7417 - loss: 0.7192 - val_accuracy: 0.7545 - val_loss: 0.5975 - learning_rate: 1.0000e-04
Epoch 2/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 496s 4s/step - accuracy: 0.8886 - loss: 0.3040 - val_accuracy: 0.8714 - val_loss: 0.3734 - learning_rate: 1.0000e-04
Epoch 3/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 483s 3s/step - accuracy: 0.9201 - loss: 0.2172 - val_accuracy: 0.9116 - val_loss: 0.2404 - learning_rate: 1.0000e-04
Epoch 4/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 480s 3s/step - accuracy: 0.9342 - loss: 0.1771 - val_accuracy: 0.9000 - val_loss: 0.2891 - learning_rate: 1.0000e-04
Epoch 5/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 488s 3s/step - accuracy: 0.9513 - loss: 0.1357 - val_accuracy: 0.9196 - val_loss: 0.2505 - learning_rate: 1.0000e-04
Epoch 6/30
140/140 ━━━━━━━━━━━━━━━━━━━━ 487s 3s/step - accuracy: 0.9560 - loss: 0.1226 - 